# 33 — bf16 baseline: composition families vs GDS variants (n=2 and n=3)

The new working baseline, on unquantized Llama (bf16, sharded) with the
validated bf16 pool of notebook 32, extending the notebook-28 nested design
(10 ρ-spanning pairs, 30 triples at distant/mid/close entanglement, both
arms fresh) with the literature-motivated methods:

| continuity | GDS (generalized difference subspace) | soft |
|---|---|---|
| blend, vote, schedule | **gds-blend** — projector composite Π((I−MMᵀ)ΣFᵢ), token-ordered gauge anchored by the frame sum; **gds-vote / gds-schedule** — faithful Fukui usage: each concept individually projected off the common subspace (Π_D Cᵢ, re-orthonormalized), per-concept scoring kept | **soft-vote** — z-scored sum with a softplus barrier below threshold (smooth F2.c; Prism-Δ's continuous-weighting lesson) |

7 methods × 40 groups × 10 prompts. Design notes:

- The common/difference split uses `common_threshold = 1.05` (eigenvalues of
  G = ΣFᵢFᵢᵀ above it are shared directions; 1 is the theoretical boundary).
- Since the projector is linear, `gds-blend` equals a blend of the projected
  guides *without* per-concept re-orthonormalization — so the gds-blend vs
  gds-vote/schedule contrast isolates re-orthonormalization + per-concept
  scoring, the "projector vs faithful" comparison.
- **Evaluation always scores against the original concepts**, never the
  preprocessed guides — methods compete on the same yardstick.
- Both metric tracks (expression + angles) on every record.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.linalg.gds import gds_blend, remove_common
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.aggregators import soft_constrained_scorer
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
N_BASE_PAIRS = 10
MAX_USES_PER_CONCEPT = 2
COMMON_THRESHOLD = 1.05  # gds_split eigenvalue boundary for "shared"
SOFT_BETA = 8.0

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
POOL_NAMES = [r["synset"] for r in POOL["pool"]]
DELTA1 = {r["synset"]: r["delta"] for r in POOL["pool"]}


def rho(a: str, b: str) -> float:
    key = f"{a}|{b}" if f"{a}|{b}" in POOL["pairwise_rho"] else f"{b}|{a}"
    return POOL["pairwise_rho"][key]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/33_scaling3_bf16.jsonl")
GROUPS_PATH = Path("resources/33_scaling_groups_bf16.json")
LOG_PATH.unlink(missing_ok=True)
print(f"pool: {len(POOL_NAMES)} validated concepts")

In [ ]:
# 10 base pairs evenly spaced along the pool's rho spectrum, each concept
# used at most twice; each pair extended with a third concept at three
# entanglement strata (identical logic to notebook 28)
all_pairs = sorted(
    (
        {"a": a, "b": b, "pair_rho": rho(a, b)}
        for i, a in enumerate(POOL_NAMES)
        for b in POOL_NAMES[i + 1 :]
    ),
    key=lambda p: p["pair_rho"],
)
targets = [
    all_pairs[round(i * (len(all_pairs) - 1) / (N_BASE_PAIRS - 1))]["pair_rho"]
    for i in range(N_BASE_PAIRS)
]
PAIRS, uses = [], {}
for t in targets:
    for p in sorted(all_pairs, key=lambda p: abs(p["pair_rho"] - t)):
        key = (p["a"], p["b"])
        if key in {(q["a"], q["b"]) for q in PAIRS}:
            continue
        if uses.get(p["a"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        if uses.get(p["b"], 0) >= MAX_USES_PER_CONCEPT:
            continue
        PAIRS.append(p)
        uses[p["a"]] = uses.get(p["a"], 0) + 1
        uses[p["b"]] = uses.get(p["b"], 0) + 1
        break
if len(PAIRS) < N_BASE_PAIRS:
    raise RuntimeError("usage cap prevented an even pair spread")

TRIPLES = []
for p in PAIRS:
    a, b = p["a"], p["b"]
    ranked = sorted(
        (c for c in POOL_NAMES if c not in (a, b)),
        key=lambda c: max(rho(c, a), rho(c, b)),
    )
    picks = {"distant": ranked[0], "mid": ranked[len(ranked) // 2], "close": ranked[-1]}
    for add_regime, c in picks.items():
        TRIPLES.append(
            {
                "a": a,
                "b": b,
                "c": c,
                "add_regime": add_regime,
                "add_rho": max(rho(c, a), rho(c, b)),
                "mean_rho": (rho(a, b) + rho(a, c) + rho(b, c)) / 3,
                "pair_rho": p["pair_rho"],
            }
        )

GROUPS_PATH.write_text(
    json.dumps(
        {
            "config": {
                "pool": "resources/32_concept_pool_bf16.json",
                "n_base_pairs": N_BASE_PAIRS,
                "max_uses_per_concept": MAX_USES_PER_CONCEPT,
                "common_threshold": COMMON_THRESHOLD,
                "soft_beta": SOFT_BETA,
            },
            "pairs": PAIRS,
            "triples": TRIPLES,
        },
        indent=1,
    )
)
print(
    f"{len(PAIRS)} pairs (rho {PAIRS[0]['pair_rho']:.2f}.."
    f"{PAIRS[-1]['pair_rho']:.2f}), {len(TRIPLES)} triples -> {GROUPS_PATH}"
)

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
prompts = [chat(u, a) for u, a in QUESTIONS]

harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

used = sorted({s for g in PAIRS + TRIPLES for s in (g["a"], g["b"], g.get("c")) if s})
concept = {name: fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS) for name in used}
# sharded model: cached loads land on model.device but freshly computed
# concepts land on the lm_head's device — normalize before any stacking
DEVICE = fur.model.device
concept = {
    n: Concept(synset=c.synset, tensor=c.tensor.to(DEVICE)) for n, c in concept.items()
}
print(f"{len(concept)} concepts built")


def stack_frames(cs: list[Concept]) -> torch.Tensor:
    return torch.cat(Concept._pad_to_common_rank(cs))


def gds_guides(cs: list[Concept]) -> list[Concept]:
    """Faithful GDS: each concept projected off the group's common subspace."""
    cleaned = remove_common(stack_frames(cs), common_threshold=COMMON_THRESHOLD)
    guides = [
        Concept(synset=c.synset, tensor=cleaned[i : i + 1]) for i, c in enumerate(cs)
    ]
    if any(int(g.rank) == 0 for g in guides):
        raise RuntimeError("a concept collapsed entirely into the common subspace")
    return guides


def gds_blend_guide(cs: list[Concept]) -> Concept:
    """Projector composite: polar((I - MM^T) sum F_i), gauge from the sum."""
    blended = gds_blend(stack_frames(cs), common_threshold=COMMON_THRESHOLD)
    name = " + ".join(c.synset for c in cs) + " (gds)"
    return Concept(synset=name, tensor=blended.unsqueeze(0))


def run_group(group: dict) -> None:
    synsets = [s for s in (group["a"], group["b"], group.get("c")) if s]
    cs = [concept[s] for s in synsets]
    methods = {
        "blend": lambda: fur.generate_with_topk_guide(
            prompts, guide=Concept.average(cs), k=K, steps=STEPS
        ),
        "vote": lambda: fur.generate_with_topk_multi_guide(
            prompts, guides=cs, k=K, steps=STEPS, normalize="zscore"
        ),
        "schedule": lambda: fur.generate_with_topk_multi_guide(
            prompts, guides=cs, k=K, steps=STEPS, scorer=RoundRobin()
        ),
        "gds-blend": lambda: fur.generate_with_topk_guide(
            prompts, guide=gds_blend_guide(cs), k=K, steps=STEPS
        ),
        "gds-vote": lambda: fur.generate_with_topk_multi_guide(
            prompts, guides=gds_guides(cs), k=K, steps=STEPS, normalize="zscore"
        ),
        "gds-schedule": lambda: fur.generate_with_topk_multi_guide(
            prompts, guides=gds_guides(cs), k=K, steps=STEPS, scorer=RoundRobin()
        ),
        "soft-vote": lambda: fur.generate_with_topk_multi_guide(
            prompts,
            guides=cs,
            k=K,
            steps=STEPS,
            normalize="zscore",
            scorer=soft_constrained_scorer(beta=SOFT_BETA),
        ),
    }
    for method, generate in methods.items():
        torch.cuda.empty_cache()
        texts, probe = generate()
        harness.evaluate(
            prompts,
            texts,
            synsets,
            config={
                "model": MODEL_KEY,
                "method": method,
                "n": len(synsets),
                "common_threshold": COMMON_THRESHOLD,
                "soft_beta": SOFT_BETA,
                **group,
            },
            probe=probe,
            baseline_texts=baseline_texts,
            concepts=dict(zip(synsets, cs)),  # always score the ORIGINALS
            angles=True,
        )
    label = " + ".join(s.split(".")[0] for s in synsets)
    print(f"n={len(synsets)} {label} ({group.get('add_regime', 'pair')}): done")


for group in [*PAIRS, *TRIPLES]:
    run_group(group)

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert {r["config"]["n"] for r in parsed} == {2, 3}

rows = []
for r in parsed:
    c = r["config"]
    synsets = [s for s in (c["a"], c["b"], c.get("c")) if s]
    deltas = [r["expression"][s]["delta"] for s in synsets]
    cosines = [r["angles"][s]["delta"]["token_cos"] for s in synsets]
    rows.append(
        {
            "method": c["method"],
            "n": c["n"],
            "joint": min(deltas),
            "all_positive": min(deltas) > 0,
            "joint_cos": min(cosines),
            "all_positive_cos": min(cosines) > 0,
            "add_regime": c.get("add_regime"),
            "mean_rho": c.get("mean_rho", c["pair_rho"]),
            "pair_rho": c["pair_rho"],
            "ppl_ratio": r["ppl_ratio"],
        }
    )
results = pd.DataFrame(rows)
print(f"gate: {results.groupby('n').size().to_dict()} records")

summary = (
    results.groupby(["method", "n"])
    .agg(
        joint=("joint", "mean"),
        all_positive=("all_positive", "mean"),
        joint_cos=("joint_cos", "mean"),
        all_positive_cos=("all_positive_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n_runs=("joint", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/33_scaling3_bf16_summary.csv", index=False)
summary.round(4)

In [ ]:
order = [
    "blend",
    "gds-blend",
    "vote",
    "gds-vote",
    "soft-vote",
    "schedule",
    "gds-schedule",
]
colors = {
    "blend": "tab:blue",
    "gds-blend": "navy",
    "vote": "tab:orange",
    "gds-vote": "sienna",
    "soft-vote": "gold",
    "schedule": "tab:green",
    "gds-schedule": "darkgreen",
}
ns = [2, 3]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, (metric, label) in zip(
    axes,
    (
        ("joint", "mean joint expression gain"),
        ("joint_cos", "mean joint token-cosine gain"),
        ("all_positive_cos", "share with ALL concepts improved (cosine)"),
    ),
):
    for j, method in enumerate(order):
        vals = [
            summary[(summary["method"] == method) & (summary["n"] == n)][metric].iloc[0]
            for n in ns
        ]
        offset = (j - (len(order) - 1) / 2) * 0.11
        ax.bar(
            [i + offset for i in range(len(ns))],
            vals,
            width=0.1,
            color=colors[method],
            label=method,
        )
    ax.set_xticks(range(len(ns)), [f"n={n}" for n in ns])
    ax.set_ylabel(label)
    ax.axhline(0, color="gray", lw=0.8)
axes[0].legend(fontsize=7)
axes[2].axhline(0.5, color="gray", ls="--", lw=0.8)
plt.suptitle("bf16 baseline: composition families vs GDS variants and soft vote")
plt.tight_layout()
plt.savefig("resources/33_scaling3_bf16.png", dpi=150)
plt.show()

print("raw -> gds (same family, mean joint):")
for n in ns:
    sub = summary[summary["n"] == n].set_index("method")
    for fam in ("blend", "vote", "schedule"):
        raw, gds = sub.loc[fam], sub.loc[f"gds-{fam}"]
        print(
            f"  n={n} {fam:9s} joint {raw['joint']:+.3f} -> {gds['joint']:+.3f}"
            f"   all-improved(cos) {raw['all_positive_cos']:.0%} -> "
            f"{gds['all_positive_cos']:.0%}"
        )
    soft, vote = sub.loc["soft-vote"], sub.loc["vote"]
    print(f"  n={n} soft-vote joint {soft['joint']:+.3f} (vote {vote['joint']:+.3f})")

print("\nranking by joint_cos:")
for n in ns:
    ranked = summary[summary["n"] == n].sort_values("joint_cos", ascending=False)
    print(f"  n={n}: " + " > ".join(ranked["method"]))

In [ ]:
# GDS's motivating prediction: common-removal should help most where the
# common component is large — the high-rho half of the groups.
median_rho = results["mean_rho"].median()
results["rho_half"] = results["mean_rho"] > median_rho
pivot = results.groupby(["rho_half", "method"])["joint"].mean().unstack(0)
pivot.columns = ["low-rho half", "high-rho half"]
for fam in ("blend", "vote", "schedule"):
    gain_low = pivot.loc[f"gds-{fam}", "low-rho half"] - pivot.loc[fam, "low-rho half"]
    gain_high = (
        pivot.loc[f"gds-{fam}", "high-rho half"] - pivot.loc[fam, "high-rho half"]
    )
    print(f"{fam:9s} gds gain: low-rho {gain_low:+.3f}   high-rho {gain_high:+.3f}")
pivot.round(3)

In [ ]:
def answer(text: str) -> str:
    marker = "assistant<|end_header_id|>"
    return text.split(marker)[-1].strip() if marker in text else text


for method in ("blend", "gds-blend", "gds-vote", "soft-vote"):
    for r in parsed:
        c = r["config"]
        if (
            c["method"] == method
            and c.get("add_regime") == "close"
            and "short story" in r["prompt"]
        ):
            deltas = {
                k.split(".")[0]: round(v["delta"], 2)
                for k, v in r["expression"].items()
            }
            print(f"[{method} / close triple] {deltas}")
            print("   ", answer(r["text"])[:170], "\n")
            break